# Demo 2：Reaction to Crystallization

任务：`reaction-to-crystallization`。本 Demo 展示上游反应动力学如何通过组成与过饱和度传播到结晶产率、纯度和粒度分布，是一个组合式物理化学 world model 问题。

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'chemworld').exists()), None)
if ROOT is None:
    raise RuntimeError('请从 ChemWorld 仓库内启动 notebook')
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
du = importlib.import_module('notebooks.task_demos.demo_utils')

TASK_ID = 'reaction-to-crystallization'
SEED = 0

## 1. 公开任务合同

公开合同规定反应检测、晶种、受控冷却、浆料检测、过滤和终检的可用语义。以下 Demo 不读取 hidden state、速率常数或成核参数。

In [2]:
card = du.task_card(TASK_ID)
pd.DataFrame({
    'field': ['task_id', 'motivation', 'budget', 'episode_mode', 'success_metrics'],
    'value': [card['task_id'], card['scientific_motivation'], card['budget'], card['episode_mode'], card['reward_leaderboard_metric']['success_metrics']],
})

,field,value
0,task_id,reaction-to-crystallization
1,motivation,Use reaction assays to adapt a seeded cooling ...
2,budget,72
3,episode_mode,campaign
4,success_metrics,"[score, crystal_yield, crystal_purity, crystal..."


## 2. 构造候选干预

候选向量共同决定温度、时间、投料、催化条件、晶种量、冷却终点和冷却时间。公开 adapter 保证它们被编译成任务允许的完整实验。

In [3]:
vectors = du.standard_probe_vectors(TASK_ID)
mid_recipe = du.recipe_frame(TASK_ID, vectors['mid'])
mid_recipe

,step,operation,payload
0,1,add_solvent,"{'operation': 'add_solvent', 'volume_L': 0.025..."
1,2,add_reagent,"{'operation': 'add_reagent', 'amount_mol': 0.0..."
2,3,add_catalyst,"{'operation': 'add_catalyst', 'catalyst_amount..."
3,4,heat,"{'operation': 'heat', 'target_temperature_K': ..."
4,5,quench,{'operation': 'quench'}
5,6,measure,"{'operation': 'measure', 'instrument': 'hplc'}"
6,7,seed_crystals,"{'operation': 'seed_crystals', 'seed_mass_g': ..."
7,8,cool_crystallize,"{'operation': 'cool_crystallize', 'target_temp..."
8,9,measure,"{'operation': 'measure', 'instrument': 'hplc'}"
9,10,filter_crystals,{'operation': 'filter_crystals'}


## 3. 执行并读取反馈

比较 `crystal_yield`、`crystal_purity`、`crystal_csd_quality` 和 `crystal_fines_fraction`。同一最终得分可能来自不同的产率—纯度—粒度权衡。

In [4]:
comparison = du.compare_vectors(TASK_ID, vectors, seed=SEED)
columns = ['candidate', 'leaderboard_score', 'crystal_yield', 'crystal_purity', 'crystal_csd_quality', 'crystal_fines_fraction', 'all_actions_valid']
comparison[[column for column in columns if column in comparison]]

,candidate,leaderboard_score,crystal_yield,crystal_purity,crystal_csd_quality,crystal_fines_fraction,all_actions_valid
0,low,0.375489,0.448629,0.982604,0.462966,1.000000,True
1,mid,0.436261,0.541777,0.982912,0.557991,0.780568,True
2,high,0.354669,0.609051,0.985155,0.478158,0.942438,True


### 查看反应前后与结晶后的测量证据

公开 `processed_estimate` 的时间顺序可以支持跨阶段预测误差分析。

In [5]:
run = du.run_vector(TASK_ID, vectors['mid'], seed=SEED)
measurement_feedback = du.measurement_trace(run)
measurement_feedback

,step,operation,reward,leaderboard_score,observed_keys,processed_estimate
0,6,measure,0.054108,NaN,"yield, selectivity, byproduct_signal, crystal_...","{'yield': 0.40546153015058223, 'selectivity': ..."
1,9,measure,0.188792,NaN,"yield, selectivity, byproduct_signal, crystal_...","{'yield': 0.4002589604598178, 'selectivity': 0..."
2,12,measure,0.193361,0.436261,"yield, selectivity, conversion, byproduct_sign...","{'yield': 0.3993259866821407, 'selectivity': 0..."


## 4. 同一干预，不同隐藏规律

教师端用 `rate_law_family` 生成 World B。公共 recipe 不变，变化只能通过反应和结晶反馈显现；正式 Agent 不会看到机制名称。

In [6]:
paired_worlds = du.compare_hidden_worlds(
    TASK_ID, vectors['mid'], mechanism_mode='rate_law_family', seed=SEED
)
paired_worlds

,opaque_world,score,crystal_yield,crystal_purity,crystal_size,crystal_csd_quality,crystal_fines_fraction,leaderboard_score,cost,safety_risk,all_actions_valid
0,World A,None,0.541777,0.982912,0.043507,0.557991,0.780568,0.436261,0.0,None,True
1,World B,None,0.542580,0.982988,0.042366,0.552999,0.793377,0.428751,0.0,None,True


## 5. 留给 Agent 的能力问题

需要区分：结果差异来自上游反应组成，还是来自下游成核/生长响应。环境不替 Agent 训练或更新模型。

In [7]:
capability_probe = {
    'current_evidence': comparison.to_dict(orient='records'),
    'prediction_query': '预测一个未执行冷却条件下的 yield/purity/CSD 联合分布。',
    'next_intervention_query': '选择能区分动力学错误与结晶机理错误的下一次完整实验。',
    'evaluation_note': '检验跨阶段可执行预测，而不是只比较最终 leaderboard_score。',
}
capability_probe

{'current_evidence': [{'candidate': 'low',
   'score': None,
   'crystal_yield': 0.4486290766503891,
   'crystal_purity': 0.9826040657100683,
   'crystal_size': 0.02862537684459947,
   'crystal_csd_quality': 0.46296630396634914,
   'crystal_fines_fraction': 1.0,
   'leaderboard_score': 0.3754887044854116,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 12},
  {'candidate': 'mid',
   'score': None,
   'crystal_yield': 0.5417773691401575,
   'crystal_purity': 0.9829121939367824,
   'crystal_size': 0.04350656824479965,
   'crystal_csd_quality': 0.5579909155643324,
   'crystal_fines_fraction': 0.7805680353231376,
   'leaderboard_score': 0.4362611199820082,
   'cost': 0.0,
   'safety_risk': None,
   'all_actions_valid': True,
   'operation_count': 12},
  {'candidate': 'high',
   'score': None,
   'crystal_yield': 0.6090511757926285,
   'crystal_purity': 0.9851547365755134,
   'crystal_size': 0.039875586082113376,
   'crystal_csd_quality': 0.478157